In [94]:
import numpy as np
import os,sys
sys.path.append(os.path.realpath('../lib'))
from graph_multimodalsentence import  graph_colors,draw_rel,draw_transp,Graph,wl_labeling,find_thresh,sp_to_adjency
from ot_distances import Fused_Gromov_Wasserstein_distance
from sentence_transformers import SentenceTransformer
from FGW import fgw_barycenters
import seaborn as sns

In [95]:
method='shortest_path_weight'
embedding = SentenceTransformer('sentence-transformers/bert-base-nli-mean-tokens')
def create_graph_from_data(x,embedding):
    graphs=Graph()
    graphs.add_sentence(x,embedding)
    return graphs

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7766.00it/s]


In [96]:
import pandas as pd
data = pd.read_csv('../data_with_text_tag\generated_instructions_with_intention_explanation_full_benevolence_gender_neutral_gpt_qa.csv')
data_high = data[data['benevolence'] == 'Low']
graphs_high=[create_graph_from_data(x,embedding) for x in data_high.instruction_with_intention]

Talking
Pointing
Talking
Pointing
Talking 3
Acknowledging
Talking
Pointing
Talking 3
Pointing
Talking
Pointing
Talking 3
Pointing
Talking 3
Pointing
Shrugging
Talking 3
Shrugging
Talking 3
Shrugging
Pointing
Talking
Pointing
Shrugging
Talking 3
Shrugging
Talking 3
Shrugging
Talking 3
Shrugging
Pointing
Shrugging
Pointing
Shrugging
Pointing
Shrugging
Pointing
Shrugging
Pointing
Shrugging
Talking 3
Talking
Head Nod Yes
Talking
Head Nod Yes
Talking 3
Pointing
Talking 3
Pointing
Talking 3
Acknowledging
Talking
Pointing
Shrugging
Pointing
Talking
Pointing
Shrugging
Talking 3
Pointing
Shrugging
Shrugging
Talking 3
Shrugging
Talking 3
Acknowledging
Talking 3
Head Nod Yes
Shrugging
Pointing
Shrugging
Pointing
Talking 3
Head Nod Yes
Shrugging
Pointing
Talking 3
Pointing
Talking 3
Acknowledging
Talking 3
Acknowledging
Talking
Pointing
Shrugging
Pointing
Talking 3
Pointing
Shrugging
Talking 3
Head Nod Yes
Shrugging
Talking 3
Head Nod Yes
Talking 3
Head Nod Yes
Shrugging
Pointing
Talking 3
Pointin

In [97]:
def label_edge(graph,edge):
    # label edges with the labels of the nodes they connect
    return (graph.nodes()[edge[0]]['label'],graph.nodes()[edge[1]]['label'])

def indice_edge(graph,edge):
    try:
        source_indice = list([graph.nodes()[i]['label'] for i in graph.nodes()]).index(edge[0])
        target_indice = list([graph.nodes()[i]['label'] for i in graph.nodes()]).index(edge[1])
        return (source_indice,target_indice)
    except ValueError:
        return (None, None)

# find edges often recurring in a set of graphs
def patternfinder(graphs,threshold=0.01):
    # find edges often recurring in a set of graphs
    edges=[]
    for graph in graphs:
        l=[e for e in graph.edges()]
        for e in l:
            e_name = label_edge(graph,e)
            if e_name not in edges:
                edges.append(e_name)
       
    edges=list(set(edges))
    edge_count=[sum([1 for graph in graphs if indice_edge(graph,e) in graph.edges()]) for e in edges]
    edge_count=np.array(edge_count)/len(graphs)
    pattern_edges=[edges[i] for i in range(len(edges)) if edge_count[i]>threshold]
    return pattern_edges

In [98]:
def two_edges_patternfinder(graphs,threshold=0.01):
    # find edges often recurring pair of sequential edges in a set of graphs
    
    edges=[]
    for graph in graphs:
        l=[e for e in graph.edges()]
        for e1 in l:
            for e2 in l:
                if e1 != e2:
                    s1,t1 = label_edge(graph,e1)
                    s2,t2 = label_edge(graph,e2)
                    e_name = ((s1,t1),(s2,t2))
                    if e_name not in edges:
                        edges.append(e_name)

    edges=list(set(edges))
    edge_count=[sum([1 for graph in graphs if indice_edge(graph,e1) in graph.edges() and indice_edge(graph,e2) in graph.edges()]) for e1,e2 in edges]
    edge_count=np.array(edge_count)/len(graphs)
    
    
    

In [99]:
def print_patternfinder(graphs,threshold=0.5):
    pattern_edges=patternfinder(graphs,threshold)
    print("Pattern edges found in the graphs:")
    for edge in pattern_edges:
        print(edge)

In [100]:
def print_two_edges_patternfinder(graphs,threshold=0.5):
    pattern_edges=two_edges_patternfinder(graphs,threshold)
    print("Pattern edges found in the graphs:")
    for edge in pattern_edges:
        print(edge)

In [101]:
print_patternfinder(graphs_high,threshold=0.5)

Pattern edges found in the graphs:
('f:confidence', 't:Impersonal.Pronoun')
('f:neutral', 'f:confidence')
('f:neutral', 't:Positive.Emotion')
('i:thoughtful', 't:Positive.Emotion')
('f:neutral', 't:Impersonal.Pronoun')
('g:Pointing', 't:Impersonal.Pronoun')
('f:confidence', 'g:Pointing')
('f:neutral', 'i:thoughtful')
